In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import folium
from folium.plugins import Fullscreen
from branca.element import Template, MacroElement
import geopandas as gpd
import fiona
from shapely.geometry import shape
import os
import json
import urllib.parse
import glob

out_dir = "./output"  # [CENSORED: Set your output directory here]
input_dir = "./input_data"  # [CENSORED: Set your input site data directory here]
clutter_dir = "./clutter_data"  # [CENSORED: Set your MapInfo .TAB clutter directory here]

print("1. Loading base Mushroom data via glob...")
excel_files = glob.glob(os.path.join(input_dir, "*.xlsx"))
if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {input_dir}")
    
input_mush = excel_files[0]
print(f"   -> Using {input_mush}")

df_all = pd.read_excel(input_mush, sheet_name='Site Classification')

df_all.rename(columns={
    'SITE_ID': 'site_id',
    'SITE_NAME': 'site_name',
    'LATITUDE': 'lat',
    'LONGITUDE': 'lon'
}, inplace=True)

def map_category(tech_str):
    if pd.isna(tech_str): return None
    tech_str = str(tech_str).upper()
    has_21 = 'NR2100' in tech_str
    has_26 = 'NR26G' in tech_str or 'NR2600' in tech_str or 'NR26' in tech_str
    
    if has_21 and has_26: return 'NR21 & NR26'
    if has_21: return 'NR21 Only'
    if has_26: return 'NR26 Only'
    if '1800' in tech_str or '2100' in tech_str or '900' in tech_str: return 'LTE Only'
    return None

df_all['Category'] = df_all['TECHNOLOGY'].apply(map_category)
df_all = df_all.dropna(subset=['Category', 'lat', 'lon']).reset_index(drop=True)

df_all['Is_IBS'] = df_all['SITE_TYPE'].str.contains('IBS', na=False)

print(f"   -> Loaded {len(df_all)} total sites (NR + LTE).")

print("1.5. Spatial Join with MapInfo Clutter Polygons...")
try:
    clutter_files = glob.glob(os.path.join(clutter_dir, "*.TAB"), recursive=True)
    if not clutter_files:
        clutter_files = glob.glob(os.path.join(clutter_dir, "*.tab"), recursive=True)
    
    if clutter_files:
        print(f"   -> Found clutter file: {clutter_files[0]}")
        records = []
        with fiona.open(clutter_files[0]) as src:
            tab_crs = src.crs
            for f in src:
                if f['geometry'] is not None:
                    records.append({
                        'geometry': shape(f['geometry']),
                        'Morpho': f['properties'].get('Morpho')
                    })
        gdf_clutter = gpd.GeoDataFrame(records, crs=tab_crs)
        if getattr(gdf_clutter, 'crs', None) is None:
            gdf_clutter.set_crs(epsg=4326, inplace=True)
        else:
            gdf_clutter = gdf_clutter.to_crs(epsg=4326)
            
        gdf_sites = gpd.GeoDataFrame(df_all, geometry=gpd.points_from_xy(df_all.lon, df_all.lat), crs="EPSG:4326")
        gdf_joined = gpd.sjoin(gdf_sites, gdf_clutter, how='left', predicate='intersects')
        gdf_joined = gdf_joined[~gdf_joined.index.duplicated(keep='first')]
        df_all['CLUTTER'] = gdf_joined['Morpho']
        print("   -> Spatial join successful.")
    else:
        print("   -> No .TAB clutter files found. Using existing CLUTTER column.")
except Exception as e:
    print(f"   -> Error in spatial join: {e}")

print("2. Running Spatial Clustering & Nearest Neighbors...")
df_all['lat_rad'] = np.radians(df_all['lat'])
df_all['lon_rad'] = np.radians(df_all['lon'])
coords = df_all[['lat_rad', 'lon_rad']].values

epsilon = 20 / 6371000
db = DBSCAN(eps=epsilon, min_samples=2, metric='haversine').fit(coords)
df_all['Coloc_Cluster'] = db.labels_

remarks = []
for label in df_all['Coloc_Cluster']:
    if label != -1:
        remarks.append('Colocated (<=20m)')
    else:
        remarks.append('-')
df_all['Remark'] = remarks

nbrs = NearestNeighbors(n_neighbors=20, metric='haversine').fit(coords)
distances, indices = nbrs.kneighbors(coords)

def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    initial_bearing = np.arctan2(x, y)
    return (np.degrees(initial_bearing) + 360) % 360

clutter_thresholds = {
    'DENSE URBAN': 500,
    'URBAN': 1000,
    'SUB URBAN': 2000,
    'RURAL': 5000
}

print("3. Analyzing NR21 Mushrooms...")
is_nr_mushroom = []
is_nr_critical = []
for i in range(len(df_all)):
    if df_all.loc[i, 'Category'] != 'NR21 Only' or df_all.loc[i, 'Is_IBS']:
        is_nr_mushroom.append(False)
        is_nr_critical.append(False)
        continue
        
    c_clutter = str(df_all.loc[i, 'CLUTTER']).upper().strip()
    max_radius = clutter_thresholds.get(c_clutter, 2000)
    
    neighbor_idx = indices[i][1:]
    neighbor_dist = distances[i][1:]
    
    upgraded_geom = []
    upgraded_crit = []
    
    for j, idx in enumerate(neighbor_idx):
        if df_all.loc[idx, 'Category'] in ['NR21 & NR26', 'NR26 Only']:
            upgraded_geom.append(idx)
            dist_m = neighbor_dist[j] * 6371000
            if dist_m <= max_radius:
                upgraded_crit.append(idx)
                
    center_lat = df_all.loc[i, 'lat']
    center_lon = df_all.loc[i, 'lon']
    
    site_is_mushroom = False
    if len(upgraded_geom) >= 3:
        bearings = [calculate_bearing(center_lat, center_lon, df_all.loc[idx, 'lat'], df_all.loc[idx, 'lon']) for idx in upgraded_geom]
        bearings.sort()
        gaps = [bearings[k+1] - bearings[k] for k in range(len(bearings) - 1)]
        if len(bearings) > 1: gaps.append(360 - bearings[-1] + bearings[0])
        else: gaps.append(360)
        
        if max(gaps) <= 210:
            site_is_mushroom = True
            
    is_nr_mushroom.append(site_is_mushroom)
    
    site_is_crit = False
    if len(upgraded_crit) >= 3:
        bearings_crit = [calculate_bearing(center_lat, center_lon, df_all.loc[idx, 'lat'], df_all.loc[idx, 'lon']) for idx in upgraded_crit]
        bearings_crit.sort()
        gaps_crit = [bearings_crit[k+1] - bearings_crit[k] for k in range(len(bearings_crit) - 1)]
        if len(bearings_crit) > 1: gaps_crit.append(360 - bearings_crit[-1] + bearings_crit[0])
        else: gaps_crit.append(360)
        
        if max(gaps_crit) <= 210:
            site_is_crit = True
            
    is_nr_critical.append(site_is_crit)

df_all['Is_NR_Mushroom'] = is_nr_mushroom
df_all['Is_NR_Critical'] = is_nr_critical
df_all.loc[df_all['Is_NR_Mushroom'], 'Category'] = 'NR21 Mushroom'
df_all.loc[df_all['Is_NR_Critical'], 'Category'] = 'Critical NR21 Mushroom'

print("4. Analyzing LTE Mushrooms...")
is_lte_mushroom = []
is_lte_critical = []
for i in range(len(df_all)):
    if df_all.loc[i, 'Category'] != 'LTE Only' or df_all.loc[i, 'Is_IBS']:
        is_lte_mushroom.append(False)
        is_lte_critical.append(False)
        continue
        
    c_clutter = str(df_all.loc[i, 'CLUTTER']).upper().strip()
    max_radius = clutter_thresholds.get(c_clutter, 2000)
    
    neighbor_idx = indices[i][1:]
    neighbor_dist = distances[i][1:]
    
    upgraded_geom = []
    upgraded_crit = []
    
    for j, idx in enumerate(neighbor_idx):
        if df_all.loc[idx, 'Category'] in ['NR21 & NR26', 'NR26 Only', 'NR21 Only', 'NR21 Mushroom', 'Critical NR21 Mushroom']:
            upgraded_geom.append(idx)
            dist_m = neighbor_dist[j] * 6371000
            if dist_m <= max_radius:
                upgraded_crit.append(idx)
                
    center_lat = df_all.loc[i, 'lat']
    center_lon = df_all.loc[i, 'lon']
    
    site_is_mushroom = False
    if len(upgraded_geom) >= 3:
        bearings = [calculate_bearing(center_lat, center_lon, df_all.loc[idx, 'lat'], df_all.loc[idx, 'lon']) for idx in upgraded_geom]
        bearings.sort()
        gaps = [bearings[k+1] - bearings[k] for k in range(len(bearings) - 1)]
        if len(bearings) > 1: gaps.append(360 - bearings[-1] + bearings[0])
        else: gaps.append(360)
        
        if max(gaps) <= 210:
            site_is_mushroom = True
            
    is_lte_mushroom.append(site_is_mushroom)
    
    site_is_crit = False
    if len(upgraded_crit) >= 3:
        bearings_crit = [calculate_bearing(center_lat, center_lon, df_all.loc[idx, 'lat'], df_all.loc[idx, 'lon']) for idx in upgraded_crit]
        bearings_crit.sort()
        gaps_crit = [bearings_crit[k+1] - bearings_crit[k] for k in range(len(bearings_crit) - 1)]
        if len(bearings_crit) > 1: gaps_crit.append(360 - bearings_crit[-1] + bearings_crit[0])
        else: gaps_crit.append(360)
        
        if max(gaps_crit) <= 210:
            site_is_crit = True
            
    is_lte_critical.append(site_is_crit)

df_all['Is_LTE_Mushroom'] = is_lte_mushroom
df_all['Is_LTE_Critical'] = is_lte_critical
df_all.loc[df_all['Is_LTE_Mushroom'], 'Category'] = 'LTE Mushroom'
df_all.loc[df_all['Is_LTE_Critical'], 'Category'] = 'Critical LTE Mushroom'


print("5. Generating Maps & Dashboards...")

input_shp = os.path.join(out_dir, "NR2600 Cluster.shp")
records_shp = []
try:
    with fiona.open(input_shp) as src:
        shp_crs = src.crs
        for f in src:
            records_shp.append({'geometry': shape(f['geometry'])})
    gdf_border = gpd.GeoDataFrame(records_shp, crs=shp_crs)
    if getattr(gdf_border, 'crs', None) is None:
        gdf_border.set_crs(epsg=4326, inplace=True)
    else:
        gdf_border = gdf_border.to_crs(epsg=4326)
except Exception as e:
    gdf_border = None

colors_map = {
    'NR21 Only': '#3B82F6',
    'NR26 Only': '#10B981',
    'NR21 & NR26': '#8B5CF6',
    'NR21 Mushroom': '#F59E0B',
    'Critical NR21 Mushroom': '#EF4444',
    'LTE Only': '#6B7280', # Gray for generic LTE
    'LTE Mushroom': '#FCD34D', # Light amber/yellow
    'Critical LTE Mushroom': '#F97316' # Bright orange
}

# Unified Dashboard Logic
valid_map_cats = ['NR21 Only', 'NR26 Only', 'NR21 & NR26', 'NR21 Mushroom', 'Critical NR21 Mushroom', 'LTE Mushroom', 'Critical LTE Mushroom']
df_dash = df_all[df_all['Category'].isin(valid_map_cats)].copy()
actionable_cats = ['NR21 Mushroom', 'Critical NR21 Mushroom', 'LTE Mushroom', 'Critical LTE Mushroom']

total_sites = len(df_all)
nr_sites_df = df_all[df_all['Category'].isin(['NR21 Only', 'NR26 Only', 'NR21 & NR26', 'NR21 Mushroom', 'Critical NR21 Mushroom'])]
nr_count = len(nr_sites_df)

reg_nr_mush = len(df_all[df_all['Category'] == 'NR21 Mushroom'])
crit_nr_mush = len(df_all[df_all['Category'] == 'Critical NR21 Mushroom'])

reg_lte_mush = len(df_all[df_all['Category'] == 'LTE Mushroom'])
crit_lte_mush = len(df_all[df_all['Category'] == 'Critical LTE Mushroom'])

chart_labels = ['LTE Only', 'NR21 Only', 'NR26 Only', 'NR21 & NR26', 'LTE Mushroom', 'Critical LTE Mushroom', 'NR21 Mushroom', 'Critical NR21 Mushroom']

flowchart_content = """flowchart TD
    A[Load Site Data *.xlsx] --> B[Categorize by TECHNOLOGY]
    B --> C{{Has 5G Tech?}}
    C -- Yes --> D{{Is site 'NR21 Only'?}}
    C -- No --> E{{Is site 'LTE Only'?}}
    
    D -- No --> I[Ignore Site]
    D -- Yes --> IBS1{{Is SITE_TYPE 'IBS'?}}
    IBS1 -- Yes --> I
    IBS1 -- No --> MUSH_NR[Run NR21 Geometric Wrap Analysis]
    
    E -- No --> I
    E -- Yes --> IBS2{{Is SITE_TYPE 'IBS'?}}
    IBS2 -- Yes --> I
    IBS2 -- No --> MUSH_LTE[Run LTE Geometric Wrap Analysis]
    
    MUSH_NR --> TEST_NR{{Max Angular Gap <= 210°?}}
    TEST_NR -- No --> I
    TEST_NR -- Yes --> FLAG_NR[Flag as 'NR21 Mushroom']
    
    MUSH_LTE --> TEST_LTE{{Max Angular Gap <= 210°?}}
    TEST_LTE -- No --> I
    TEST_LTE -- Yes --> FLAG_LTE[Flag as 'LTE Mushroom']
    
    FLAG_NR --> CLUTTER_NR[Check 3GPP Clutter Distances]
    CLUTTER_NR --> CRIT_NR{{>= 3 Neighbors within Radius?}}
    CRIT_NR -- No --> KEEP_NR[Keep as 'NR21 Mushroom']
    CRIT_NR -- Yes --> PROMOTE_NR[Promote to 'Critical NR21 Mushroom']
    
    FLAG_LTE --> CLUTTER_LTE[Check 3GPP Clutter Distances]
    CLUTTER_LTE --> CRIT_LTE{{>= 3 Neighbors within Radius?}}
    CRIT_LTE -- No --> KEEP_LTE[Keep as 'LTE Mushroom']
    CRIT_LTE -- Yes --> PROMOTE_LTE[Promote to 'Critical LTE Mushroom']
    
    PROMOTE_NR --> END([End])
    KEEP_NR --> END
    PROMOTE_LTE --> END
    KEEP_LTE --> END
    I --> END"""

center_lat = df_dash['lat'].mean()
center_lon = df_dash['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles='CartoDB dark_matter', prefer_canvas=True)
Fullscreen(position='topleft', title='Full Screen', title_cancel='Exit Full Screen').add_to(m)

if gdf_border is not None:
    folium.GeoJson(
        gdf_border,
        name='Cluster Boundary',
        style_function=lambda x: {'fillColor': '#FFFFFF', 'color': '#FFFFFF', 'weight': 1, 'fillOpacity': 0.05, 'dashArray': '4, 4'}
    ).add_to(m)

for idx, row in df_dash.iterrows():
    cat = row['Category']
    color = colors_map.get(cat, 'gray')
    tooltip_html = f"<b>Site ID:</b> {row['site_id']}<br><b>Category:</b> {cat}<br><b>Clutter:</b> {row['CLUTTER']}<br><b>Remark:</b> {row['Remark']}"
    
    rad = 4
    if cat in actionable_cats:
        rad = 8
        
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=rad,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        tooltip=tooltip_html
    ).add_to(m)

coloc_clusters = df_dash[df_dash['Coloc_Cluster'] != -1]
if not coloc_clusters.empty:
    cluster_centroids = coloc_clusters.groupby('Coloc_Cluster')[['lat', 'lon']].mean()
    for _, centroid in cluster_centroids.iterrows():
        folium.Circle(
            location=[centroid['lat'], centroid['lon']],
            radius=30,
            color='#06B6D4',
            weight=2,
            fill=False,
            tooltip="Colocated Area (<=20m)"
        ).add_to(m)

legend_rows = ""
for label in chart_labels:
    col = colors_map.get(label, 'gray')
    legend_rows += f'<div style="margin-bottom: 8px;"><i style="background: {col}; width: 14px; height: 14px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%;"></i> {label}</div>'

legend_html = f'''
{{% macro html(this, kwargs) %}}
<div id="custom-legend" style="position: absolute; 
     top: 15px; right: 15px; width: 250px; height: auto; 
     border:1px solid #374151; z-index:9999; font-size:13px;
     background-color: rgba(31, 41, 55, 0.95);
     padding: 16px; border-radius: 8px;
     font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; color: #F9FAFB;
     box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3);">
     <b style="font-size: 15px; margin-bottom: 12px; display: block; color: #F9FAFB;">Unified Site Classification</b>
     {legend_rows}
     <div style="margin-top: 12px; border-top: 1px solid #4B5563; padding-top: 10px;">
        <i style="border: 2px solid #06B6D4; width: 12px; height: 12px; float: left; margin-right: 10px; margin-top: 2px; border-radius: 50%; background: transparent;"></i> Colocated (<=20m)
     </div>
</div>
{{% endmacro %}}
{{% macro script(this, kwargs) %}}
    var legend = document.getElementById("custom-legend");
    var leafletMap = document.querySelector(".leaflet-container");
    if (legend && leafletMap) {{
        leafletMap.appendChild(legend);
    }}
{{% endmacro %}}
'''
macro = MacroElement()
macro._template = Template(legend_html)
m.get_root().add_child(macro)

map_html_str = m.get_root().render()
encoded_map_html = urllib.parse.quote(map_html_str)

df_table = df_all[df_all['Category'].isin(actionable_cats)][['site_id', 'lat', 'lon', 'TECHNOLOGY', 'Category', 'Remark']].copy()
df_table['lat'] = df_table['lat'].round(6)
df_table['lon'] = df_table['lon'].round(6)
df_table['TECHNOLOGY'] = df_table['TECHNOLOGY'].fillna('-').astype(str)
table_data = df_table.to_dict(orient='records')

df_all_export = df_all[['site_id', 'lat', 'lon', 'TECHNOLOGY', 'Category', 'Remark']].copy()
df_all_export['lat'] = df_all_export['lat'].round(6)
df_all_export['lon'] = df_all_export['lon'].round(6)
df_all_export['TECHNOLOGY'] = df_all_export['TECHNOLOGY'].fillna('-').astype(str)
all_table_data = df_all_export.to_dict(orient='records')

cat_counts = df_all['Category'].value_counts().to_dict()

html_template = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Unified Mushroom Master Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.0.0"></script>
    <link href="https://cdn.jsdelivr.net/npm/gridjs/dist/theme/mermaid.min.css" rel="stylesheet" />
    <script src="https://cdn.jsdelivr.net/npm/gridjs/dist/gridjs.umd.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
    <script type="module">
      import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs';
      mermaid.initialize({{ startOnLoad: true, theme: 'dark' }});
    </script>
    
    <style>
        :root {{
            --bg-color: #111827;
            --card-bg: #1F2937;
            --text-main: #F9FAFB;
            --text-muted: #9CA3AF;
            --border-color: #374151;
            --blue: #3B82F6;
            --green: #10B981;
            --purple: #8B5CF6;
            --orange: #F59E0B;
            --red: #EF4444;
            --amber: #FCD34D;
        }}
        body {{ 
            font-family: 'Inter', -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; 
            background-color: var(--bg-color); 
            color: var(--text-main); 
            margin: 0; 
            padding: 40px; 
        }}
        .container {{ max-width: 1200px; margin: 0 auto; }}
        .header {{ text-align: center; margin-bottom: 40px; }}
        .header h1 {{ margin: 0; font-size: 36px; font-weight: 800; color: var(--text-main); letter-spacing: -0.5px; }}
        .header p {{ color: var(--text-muted); font-size: 16px; margin-top: 8px; font-weight: 500; }}
        
        .kpi-container {{ display: flex; justify-content: center; gap: 24px; margin-bottom: 40px; flex-wrap: wrap; }}
        .kpi-card {{ 
            background: var(--card-bg); 
            border-radius: 16px; 
            padding: 24px; 
            width: 230px; 
            text-align: center; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2);
            border: 1px solid var(--border-color);
        }}
        .kpi-title {{ font-size: 13px; color: var(--text-muted); text-transform: uppercase; margin-bottom: 12px; font-weight: 700; letter-spacing: 0.5px; }}
        .kpi-value {{ font-size: 42px; font-weight: 800; margin: 0; line-height: 1; }}
        
        .kpi-value.blue {{ color: var(--blue); }}
        .kpi-value.green {{ color: var(--green); }}
        .kpi-value.orange {{ color: var(--orange); }}
        .kpi-value.amber {{ color: var(--amber); }}
        .kpi-value.red {{ color: var(--red); }}
        
        .charts-container {{ display: flex; gap: 30px; justify-content: space-between; flex-wrap: wrap; margin-bottom: 40px; }}
        .chart-box {{ 
            background: var(--card-bg); 
            padding: 30px; 
            border-radius: 16px; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2); 
            border: 1px solid var(--border-color);
            flex: 1;
            min-width: 450px; 
            height: 350px;
        }}
        
        .table-container, .map-container {{
            background: var(--card-bg); 
            padding: 30px; 
            border-radius: 16px; 
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.2); 
            border: 1px solid var(--border-color);
            margin-bottom: 40px;
        }}
        .table-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
            border-bottom: 2px solid var(--bg-color);
            padding-bottom: 15px;
        }}
        .table-header h2 {{ margin: 0; font-size: 20px; color: var(--text-main); font-weight: 700; }}
        .table-badge {{ background: #451a1a; color: #EF4444; padding: 6px 14px; border-radius: 999px; font-size: 13px; font-weight: 700; border: 1px solid #7f1d1d; }}
        .table-badge.blue {{ background: #172554; color: #3B82F6; border-color: #1e3a8a; }}
        .table-badge.orange {{ background: #451a1a; color: #F59E0B; border-color: #7f1d1d; }}
        
        .export-btn {{
            background-color: var(--blue);
            color: white;
            border: none;
            padding: 8px 16px;
            border-radius: 6px;
            font-family: 'Inter', sans-serif;
            font-weight: 600;
            font-size: 13px;
            cursor: pointer;
            display: flex;
            align-items: center;
            gap: 6px;
        }}
        .export-btn.secondary {{ background-color: #374151; }}
        .gridjs-wrapper {{ border: none !important; box-shadow: none !important; border-radius: 8px !important; overflow: hidden; background: var(--card-bg) !important; }}
        .gridjs-th {{ background-color: #374151 !important; color: #F9FAFB !important; font-weight: 600 !important; font-size: 13px !important; border: none !important; }}
        .gridjs-td {{ border: none !important; border-bottom: 1px solid #374151 !important; font-size: 14px; color: #F9FAFB !important; background: var(--card-bg) !important; }}
        .gridjs-search-input {{ background: #374151 !important; color: #F9FAFB !important; border-radius: 8px !important; border: 1px solid #4B5563 !important; }}
        .gridjs-pagination {{ background: var(--card-bg) !important; border-top: 1px solid #374151 !important; color: #9CA3AF !important; }}
        .gridjs-summary {{ color: #9CA3AF !important; }}
        .gridjs-pages button {{ background: #374151 !important; color: #F9FAFB !important; border: 1px solid #4B5563 !important; }}
        
        details > summary {{ list-style: none; user-select: none; }}
        details > summary::-webkit-details-marker {{ display: none; }}
    </style>
</head>
<body>

<div class="container">
    <div class="header">
        <h1>Unified Master Dashboard</h1>
        <p>Comprehensive NR & LTE Spatial Intelligence</p>
    </div>

    <div class="table-container" style="margin-bottom: 40px;">
        <details>
            <summary style="font-size: 20px; font-weight: 700; color: #F9FAFB; cursor: pointer; display: flex; align-items: center; gap: 10px;">
                <svg width="24" height="24" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" d="M19 9l-7 7-7-7"></path></svg>
                View Unified Algorithm Flowchart
            </summary>
            <div style="margin-top: 20px; padding: 20px; background-color: #111827; border-radius: 8px; display: flex; justify-content: center;">
                <div class="mermaid">
                {flowchart_content}
                </div>
            </div>
        </details>
    </div>

    <div class="kpi-container">
        <div class="kpi-card">
            <div class="kpi-title">Total Network Sites</div>
            <div class="kpi-value" style="color: var(--text-main);">{total_sites}</div>
            <div style="font-size: 11px; font-weight: 700; color: var(--text-muted); margin-top: 8px;">ALL TECHNOLOGIES</div>
        </div>
        <div class="kpi-card">
            <div class="kpi-title">Total 5G Infrastructure</div>
            <div class="kpi-value blue">{nr_count}</div>
            <div style="font-size: 11px; font-weight: 700; color: var(--blue); margin-top: 8px;">ALL NR SITES</div>
        </div>
        <div class="kpi-card" style="border-color: #b45309;">
            <div class="kpi-title" style="color: #F59E0B;">NR21 Mushrooms</div>
            <div style="display: flex; justify-content: center; align-items: baseline; gap: 8px;">
                <div style="display: flex; flex-direction: column; align-items: center;">
                    <div class="kpi-value orange" style="font-size: 34px;">{reg_nr_mush}</div>
                    <div style="font-size: 11px; font-weight: 700; color: var(--orange); margin-top: 4px;">REGULAR</div>
                </div>
                <div style="font-size: 20px; font-weight: 800; color: var(--orange);">+</div>
                <div style="display: flex; flex-direction: column; align-items: center;">
                    <div class="kpi-value red" style="font-size: 34px;">{crit_nr_mush}</div>
                    <div style="font-size: 11px; font-weight: 700; color: var(--red); margin-top: 4px;">CRITICAL</div>
                </div>
            </div>
            <div style="font-size: 11px; font-weight: 700; color: var(--text-muted); margin-top: 8px;">(MACRO ONLY)</div>
        </div>
        <div class="kpi-card" style="border-color: #b45309;">
            <div class="kpi-title" style="color: #FCD34D;">LTE Mushrooms</div>
            <div style="display: flex; justify-content: center; align-items: baseline; gap: 8px;">
                <div style="display: flex; flex-direction: column; align-items: center;">
                    <div class="kpi-value amber" style="font-size: 34px;">{reg_lte_mush}</div>
                    <div style="font-size: 11px; font-weight: 700; color: var(--amber); margin-top: 4px;">REGULAR</div>
                </div>
                <div style="font-size: 20px; font-weight: 800; color: var(--amber);">+</div>
                <div style="display: flex; flex-direction: column; align-items: center;">
                    <div class="kpi-value red" style="font-size: 34px;">{crit_lte_mush}</div>
                    <div style="font-size: 11px; font-weight: 700; color: var(--red); margin-top: 4px;">CRITICAL</div>
                </div>
            </div>
            <div style="font-size: 11px; font-weight: 700; color: var(--text-muted); margin-top: 8px;">(MACRO ONLY)</div>
        </div>
    </div>

    <div class="charts-container">
        <div class="chart-box">
            <canvas id="barChart"></canvas>
        </div>
        <div class="chart-box" style="flex: 0.6; min-width: 350px;">
            <canvas id="pieChart"></canvas>
        </div>
    </div>

    <div class="table-container">
        <div class="table-header">
            <div style="display: flex; align-items: center; gap: 16px;">
                <h2>Unified Actionable Sites</h2>
            </div>
            <div style="display: flex; gap: 10px;">
                <button onclick="exportToCSV('actionable')" class="export-btn">Export Issues Only</button>
                <button onclick="exportToCSV('all')" class="export-btn secondary">Export All Sites</button>
            </div>
        </div>
        <div id="wrapper"></div>
    </div>
    
    <div class="map-container">
        <div class="table-header">
            <div style="display: flex; align-items: center; gap: 16px;">
                <h2>Unified Geospatial Network Visualization</h2>
                <span class="table-badge blue">Hiding non-actionable LTE to preserve performance</span>
            </div>
        </div>
        <iframe id="mapFrame" allowfullscreen="true" style="width:100%; height:750px; border:none; border-radius:8px; background: #111827;"></iframe>
    </div>
</div>

<script>
    const tableData = {json.dumps(table_data)};
    const allTableData = {json.dumps(all_table_data)};
    const catCounts = {json.dumps(cat_counts)};
    const colorsMap = {json.dumps(colors_map)};
    
    const mapHtmlString = decodeURIComponent("{encoded_map_html}");
    document.getElementById('mapFrame').srcdoc = mapHtmlString;
    
    function exportToCSV(type) {{
        const headers = ['Site ID', 'Latitude', 'Longitude', 'Technology', 'Category', 'Remark'];
        const dataToExport = type === 'all' ? allTableData : tableData;
        const filename = type === 'all' ? 'Unified_Network_Sites_Export.csv' : 'Unified_Actionable_Mushroom_Sites_Export.csv';
        
        const rows = dataToExport.map(r => [r.site_id, r.lat, r.lon, r.TECHNOLOGY, r.Category, r.Remark]);
        
        let csvContent = "data:text/csv;charset=utf-8," 
            + headers.join(",") + "\\n" 
            + rows.map(e => e.join(",")).join("\\n");
            
        const encodedUri = encodeURI(csvContent);
        const link = document.createElement("a");
        link.setAttribute("href", encodedUri);
        link.setAttribute("download", filename);
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
    }}

    const labels = {json.dumps(chart_labels)};
    const values = labels.map(l => catCounts[l] || 0);
    const bgColors = labels.map(l => colorsMap[l]);

    Chart.defaults.color = '#9CA3AF';
    Chart.defaults.font.family = "'Inter', sans-serif";

    new Chart(document.getElementById('barChart'), {{
        type: 'bar',
        plugins: [ChartDataLabels],
        data: {{
            labels: labels,
            datasets: [{{
                label: 'Site Count',
                data: values,
                backgroundColor: bgColors,
                borderRadius: 6,
                borderSkipped: false
            }}]
        }},
        options: {{
            layout: {{ padding: {{ top: 20 }} }},
            responsive: true,
            maintainAspectRatio: false,
            plugins: {{ 
                legend: {{ display: false }}, 
                title: {{ display: true, text: 'Infrastructure Distribution', font: {{ size: 16, weight: 600 }}, color: '#F9FAFB', padding: {{ bottom: 20 }}, align: 'start' }},
                datalabels: {{ anchor: 'end', align: 'end', color: '#F9FAFB', font: {{ weight: 'bold', size: 13 }} }}
            }},
            scales: {{ 
                y: {{ beginAtZero: true, grid: {{ color: '#374151' }}, border: {{ display: false }} }}, 
                x: {{ grid: {{ display: false }}, border: {{ display: false }} }} 
            }}
        }}
    }});

    new Chart(document.getElementById('pieChart'), {{
        type: 'doughnut',
        data: {{
            labels: labels,
            datasets: [{{
                data: values,
                backgroundColor: bgColors,
                borderWidth: 0,
                hoverOffset: 8
            }}]
        }},
        options: {{
            responsive: true,
            maintainAspectRatio: false,
            cutout: '65%',
            plugins: {{ 
                legend: {{ position: 'right', labels: {{ padding: 20, usePointStyle: true, pointStyle: 'circle', color: '#9CA3AF' }} }},
                title: {{ display: true, text: 'Site Composition', font: {{ size: 16, weight: 600 }}, color: '#F9FAFB', padding: {{ bottom: 20 }}, align: 'start' }}
            }}
        }}
    }});

    new gridjs.Grid({{
        columns: [
            {{ id: 'site_id', name: 'Site ID' }},
            {{ id: 'lat', name: 'Latitude' }},
            {{ id: 'lon', name: 'Longitude' }},
            {{ id: 'TECHNOLOGY', name: 'Technology' }},
            {{ id: 'Category', name: 'Category' }},
            {{ id: 'Remark', name: 'Remark' }}
        ],
        data: tableData,
        search: true,
        sort: true,
        pagination: {{ enabled: true, limit: 10 }},
        language: {{ search: {{ placeholder: 'Search unified actionable sites...' }} }}
    }}).render(document.getElementById('wrapper'));
</script>
</body>
</html>
"""

out_path = os.path.join(out_dir, "Unified_Mushroom_Master_Dashboard.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html_template)
print(f"  -> Exported Unified Dashboard: {out_path}")

print("6. Exporting Summary...")
excel_out = os.path.join(out_dir, "Unified_Mushroom_Summary.xlsx")
df_all[['site_id', 'site_name', 'lon', 'lat', 'Category', 'CLUTTER', 'Remark']].to_excel(excel_out, index=False)
print(f"  -> Exported: {excel_out}")
